In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
!pip install -q transformers datasets
!pip install -q peft
!pip install -q -U bitsandbytes
!pip install -q trl

In [3]:
import os
import torch
import numpy as np
import pandas as pd
from transformers import (
    AutoModelForCausalLM,
    TrainingArguments,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
from huggingface_hub import login
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [4]:
save_dir = "/kaggle/working"

In [5]:
token = ""
login(token)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [6]:
lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

In [7]:
model_id = "mistralai/Mistral-7B-Instruct-v0.1"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [9]:
# Define default tokens
DEFAULT_PAD_TOKEN = "[PAD]"
DEFAULT_EOS_TOKEN = "</s>"
DEFAULT_BOS_TOKEN = "<s>"

special_tokens_dict = {}
if tokenizer.pad_token is None:
    special_tokens_dict['pad_token'] = DEFAULT_PAD_TOKEN
if tokenizer.eos_token is None:
    special_tokens_dict['eos_token'] = DEFAULT_EOS_TOKEN
if tokenizer.bos_token is None:
    special_tokens_dict['bos_token'] = DEFAULT_BOS_TOKEN

if special_tokens_dict:
    tokenizer.add_special_tokens(special_tokens_dict)

In [10]:
# Load model with quantization and automatic device placement
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=token
)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [11]:
# Resize the model's token embeddings after adding new tokens
model.resize_token_embeddings(len(tokenizer))

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Set tokenizer padding side
tokenizer.padding_side = "right"

In [12]:
def preprocess_function(examples):
    # Format inputs with the instruction
    inputs = [
        f"[INST] Solve the following math problem: {problem}\n"
        "Show all intermediate steps and include the final answer in LaTeX format in a box like \\boxed{{}}.[/INST]"
        for problem in examples['problem']
    ]
    # Append the solution with EOS token
    targets = [
        f"{solution}{tokenizer.eos_token}"
        for solution in examples['solution']
    ]
    # Concatenate inputs and targets
    full_texts = [inp + tgt for inp, tgt in zip(inputs, targets)]
    # Tokenize the concatenated texts
    model_inputs = tokenizer(
        full_texts,
        max_length=512,
        truncation=True,
        padding="longest",  # Use dynamic padding
        return_tensors="pt"
    )
    # Create labels by cloning input IDs
    labels = model_inputs["input_ids"].clone()

    # Mask input tokens in labels
    for i in range(len(labels)):
        input_ids = tokenizer(inputs[i], add_special_tokens=False).input_ids
        input_len = len(input_ids)
        labels[i][:input_len] = -100  # Mask the input tokens
    model_inputs["labels"] = labels
    return model_inputs

In [13]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # For causal language modeling
    pad_to_multiple_of=8  # For faster GPU performance
)

In [14]:
dataset = load_dataset("lighteval/MATH", "algebra")

0000.parquet:   0%|          | 0.00/505k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/353k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1744 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1187 [00:00<?, ? examples/s]

In [15]:
for level in range(1, 6):
    
    # Load model with quantization and automatic device placement
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        token=token
    )
    model.resize_token_embeddings(len(tokenizer))
    model = get_peft_model(model, lora_config)
    tokenizer.padding_side = "right"
    
    # Filter the dataset by level before tokenization
    level_train = dataset['train'].filter(lambda x: x['level'] == f"Level {level}")
    level_test = dataset['test'].filter(lambda x: x['level'] == f"Level {level}")
    
    print(f"Number of training samples: {len(level_train)}")
    print(f"Number of test samples: {len(level_test)}")

    # Tokenize the datasets
    level_train = level_train.map(preprocess_function, batched=True)
    level_test = level_test.map(preprocess_function, batched=True)

    # Define training arguments
    training_args = TrainingArguments(
      output_dir=os.path.join(save_dir, f"results_level_{level}"),
      per_device_train_batch_size=1,
      gradient_accumulation_steps=4,
      warmup_steps=100,
      num_train_epochs=2,
      learning_rate=1e-4,
      fp16=True, 
      logging_steps=10,
      optim="paged_adamw_8bit",
      evaluation_strategy="steps",
      eval_steps=100,
      save_steps=100,
      save_total_limit=2,
      report_to="none",  
      run_name=f"Mistral_FineTuning_Level_{level}"
    )

    # Initialize the SFTTrainer
    trainer = SFTTrainer(
      model=model,
      train_dataset=level_train,
      eval_dataset=level_test,
      peft_config=lora_config,
      args=training_args,
      data_collator=data_collator,
    )

    # Start training
    trainer.train()

    # Save the model and tokenizer after training on each level
    model_save_path = os.path.join(save_dir, f"model_level_{level}")
    trainer.save_model(model_save_path)
    tokenizer.save_pretrained(model_save_path)

    # Evaluate the model
    results = trainer.evaluate()
    print(f"Overall evaluation results: {results}")

    # Generate predictions on the test set
    model.eval()  # Set model to evaluation mode
    results_list = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    for idx in range(len(level_test)):
        sample = level_test[idx]
          # Reconstruct the input prompt
        input_text = (
              f"[INST] Solve the following math problem: {sample['problem']}\n"
              "Show all intermediate steps and include the final answer in LaTeX format in a box like \\boxed{{}}.[/INST]"
          )
        ground_truth = sample['solution']

          # Tokenize the input text
        input_ids = tokenizer.encode(input_text, return_tensors='pt').to(device)

        
        with torch.no_grad():
            output_ids = model.generate(
                  input_ids,
                  max_length=1024,
                  repetition_penalty=1.2,
                  early_stopping=True
          )
        
        try:
            predicted_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        except Exception as e:
            predicted_text = e

          # Store the results
        results_list.append({
              "problem": sample['problem'],
              "level":sample['level'],
              "ground_truth": ground_truth,
              "predicted_solution": predicted_text
          })

    # Save the test results to a CSV file
    results_df = pd.DataFrame(results_list)
    results_save_path = os.path.join(save_dir, f"test_results_level_{level}.csv")
    results_df.to_csv(results_save_path, index=False)
    print(f"Saved test results to {results_save_path}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Filter:   0%|          | 0/1744 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1187 [00:00<?, ? examples/s]

Number of training samples: 178
Number of test samples: 135


Map:   0%|          | 0/178 [00:00<?, ? examples/s]

...

/opt/conda/lib/python3.10/site-packages/peft/utils/save_and_load.py:257: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/peft/utils/save_and_load.py:257: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/peft/utils/save_and_load.py:257: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/peft/utils/save_and_load.py:257: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:615: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Overall evaluation results: {'eval_loss': 0.7883439064025879, 'eval_runtime': 338.4375, 'eval_samples_per_second': 0.907, 'eval_steps_per_second': 0.115, 'epoch': 2.0}


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attentio

Saved test results to /kaggle/working/test_results_level_5.csv
